In [1]:
!pip install stanza tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 33.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 57.8 MB/s eta 0:00:0000:0100:01
  Attempting uninst

In [2]:
import pandas as pd
from collections import Counter
from tqdm import tqdm
import stanza


# ---------- CONFIG ----------
PATH_CHUNKS = r"/kaggle/input/corpus-para-ner/chunks_etiquetados_binario.xlsx"

OUTPUT_RESULTS = r"/kaggle/working/"

# ---------- CARGAR DATOS ----------
chunks_etiquetados = pd.read_excel(PATH_CHUNKS)



In [3]:
chunks_etiquetados.head()

,chunk_id,id_doc,texto_chunk,Ciencias_ambientales_ingenieria,Ciencias_espacio,Ciencias_fisicas,Ciencias_Geografia_oceanografia,Ciencias_medicas,Ciencias_metereologia,Ciencias_naturales,...,Ciencias_tierra,Ciencia_Administracion_ciencia_investigacion,Ciencia_biologia,Ciencia_enfoque_cientifico,Ciencia_hidrologia,Ciencia_matematicas_estadistica,Ciencia_patologia,Ciencia_recursos_naturales,categorias_detectadas,etiqueta_ciencia
0,0,1,"La Coalición Colombia Partido Alianza Verde, P...",0.411230,0.368818,0.363447,0.354240,0.378154,0.343397,0.387251,...,0.392998,0.423331,0.337017,0.352550,0.328395,0.394268,0.393815,0.389038,"[('ninguna', 0)]",0
1,1,1,"al mismo tiempo lo exponen, en ciertas ocasion...",0.331604,0.304309,0.346682,0.315253,0.344347,0.323262,0.344607,...,0.318409,0.378231,0.317467,0.368125,0.306375,0.381964,0.337864,0.306510,"[('ninguna', 0)]",0
2,2,1,los acuerdos con las Farc. Anunció que no prom...,0.356805,0.311163,0.303825,0.287517,0.329738,0.290927,0.332603,...,0.292858,0.405040,0.291387,0.322885,0.297822,0.338846,0.294842,0.318955,"[('ninguna', 0)]",0
3,3,1,moratoria en la explotación tipo fracking. Y f...,0.435032,0.397881,0.372960,0.368983,0.415538,0.357000,0.410173,...,0.392092,0.445708,0.370653,0.367256,0.341832,0.404120,0.375238,0.397525,"[('ninguna', 0)]",0
4,0,2,Las interpretaciones de la historia sirven com...,0.352643,0.367811,0.375234,0.357391,0.368184,0.347770,0.392265,...,0.368177,0.428733,0.353389,0.405911,0.371109,0.412153,0.397781,0.339795,"[('ninguna', 0)]",0


In [4]:
# --- Filtrar el dataframe original ---
df_filtrado = chunks_etiquetados[['chunk_id', 'id_doc', 'texto_chunk', 
                                  'categorias_detectadas', 'etiqueta_ciencia']].copy()


In [5]:
# --- Inicializar el pipeline de Stanza ---
nlp_stanza = stanza.Pipeline(
    lang="es",
    processors="tokenize,pos,lemma",
    use_gpu=True  # o False si no tienes GPU
)


2026-01-22 08:28:32 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-01-22 08:28:32 INFO: Downloaded file to /root/stanza_resources/resources.json
2026-01-22 08:28:32 WARNING: Language es package default expects mwt, which has been added


2026-01-22 08:28:40 INFO: Loading these models for language: es (Spanish):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2026-01-22 08:28:40 INFO: Using device: cuda
2026-01-22 08:28:40 INFO: Loading: tokenize
2026-01-22 08:28:45 INFO: Loading: mwt
2026-01-22 08:28:45 INFO: Loading: pos
2026-01-22 08:28:47 INFO: Loading: lemma
2026-01-22 08:28:48 INFO: Done loading processors!


# Extraer frecuencias de POS a los textos

In [6]:
def extraer_pos_frecuencias_stanza(textos):
    """
    Procesa una lista de textos con Stanza y devuelve tres listas de diccionarios:
      - verbos_frec[i]: {lema: frecuencia}
      - adjetivos_frec[i]: {lema: frecuencia}
      - sustantivos_frec[i]: {lema: frecuencia}
    """
    verbos_frec, adjetivos_frec, sustantivos_frec = [], [], []

    for texto in tqdm(textos, desc="Procesando con Stanza"):
        try:
            if not isinstance(texto, str) or not texto.strip():
                verbos_frec.append({})
                adjetivos_frec.append({})
                sustantivos_frec.append({})
                continue
                
            doc = nlp_stanza(texto)
            verbos, adjetivos, sustantivos = [], [], []

            for sent in doc.sentences:
                for w in sent.words:
                    if not w.lemma.isalpha():
                        continue
                    if w.upos == "VERB":
                        verbos.append(w.lemma.lower())
                    elif w.upos == "ADJ":
                        adjetivos.append(w.lemma.lower())
                    elif w.upos == "NOUN":
                        sustantivos.append(w.lemma.lower())

            verbos_frec.append(dict(Counter(verbos)))
            adjetivos_frec.append(dict(Counter(adjetivos)))
            sustantivos_frec.append(dict(Counter(sustantivos)))

        except Exception as e:
            print(f"Error procesando texto: {e}")
            verbos_frec.append({})
            adjetivos_frec.append({})
            sustantivos_frec.append({})

    return verbos_frec, adjetivos_frec, sustantivos_frec

In [14]:
# --- Obtener listas de frecuencias ---
textos = df_filtrado['texto_chunk'].tolist()
verbos_frec, adjetivos_frec, sustantivos_frec = extraer_pos_frecuencias_stanza(textos)

# --- Crear DataFrames separados para cada categoría gramatical ---

# DataFrame para verbos
df_verbos = df_filtrado.copy()
df_verbos['verbos_lemas_frecuencias'] = verbos_frec
df_verbos.to_excel('verbos_stanza.xlsx', index=False)

Procesando con Stanza: 100%|██████████| 62651/62651 [3:05:33<00:00,  5.63it/s]  


In [17]:
# DataFrame para adjetivos
df_adjetivos = df_filtrado.copy()
df_adjetivos['adjetivos_lemas_frecuencias'] = adjetivos_frec
df_adjetivos.to_excel('adjetivos_stanza.xlsx', index=False)

In [18]:
# DataFrame para sustantivos
df_sustantivos = df_filtrado.copy()
df_sustantivos['sustantivos_lemas_frecuencias'] = sustantivos_frec
df_sustantivos.to_excel('sustantivos_stanza.xlsx', index=False)

# Analizar frecuencias

In [15]:
PATH_WORKING = "/kaggle/working/"

df_verbos = pd.read_excel(PATH_WORKING + "verbos_stanza.xlsx")
df_adjetivos = pd.read_excel(PATH_WORKING + "adjetivos_stanza.xlsx")
df_sustantivos = pd.read_excel(PATH_WORKING + "sustantivos_stanza.xlsx")

df_verbos

,chunk_id,id_doc,texto_chunk,categorias_detectadas,etiqueta_ciencia,verbos_lemas_frecuencias
0,0,1,"La Coalición Colombia Partido Alianza Verde, P...","[('ninguna', 0)]",0,"{'presentar': 2, 'diagnosticar': 1, 'tener': 1..."
1,1,1,"al mismo tiempo lo exponen, en ciertas ocasion...","[('ninguna', 0)]",0,"{'exponer': 1, 'presentar': 1, 'ver': 1, 'tene..."
2,2,1,los acuerdos con las Farc. Anunció que no prom...,"[('ninguna', 0)]",0,"{'anunciar': 1, 'prometer': 2, 'decir': 2, 'pr..."
3,3,1,moratoria en la explotación tipo fracking. Y f...,"[('ninguna', 0)]",0,"{'mostrar': 1, 'decir': 1, 'faltar': 1, 'tener..."
4,0,2,Las interpretaciones de la historia sirven com...,"[('ninguna', 0)]",0,"{'servir': 1, 'haber': 1, 'comprender': 1, 'ci..."
...,...,...,...,...,...,...
62646,7,13676,"así, encuentra el 90 del alimento que guardó. ...","[('ninguna', 0)]",0,"{'encontrar': 1, 'guardar': 1, 'decir': 1, 'te..."
62647,8,13676,capacidad de predecir un evento futuro. Por lo...,"[('Ciencias_naturales', 0.48311546444892883)]",1,"{'predecir': 1, 'parecer': 3, 'cumplir': 1, 'r..."
62648,9,13676,similar. Se ha encontrado que éstos le roban l...,"[('ninguna', 0)]",0,"{'encontrar': 1, 'robar': 1, 'dejar': 1, 'acer..."
62649,10,13676,está siendo parcialmente validada con experime...,"[('Ciencias_naturales', 0.5003376007080078)]",1,"{'poner': 1, 'colgar': 1, 'encontrar': 1, 'con..."


In [19]:
import ast

def asegurar_dict(x):
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return {}
    return {}

def construir_tabla_frecuencias(
    df,
    columna_diccionarios,
    columna_ciencia="etiqueta_ciencia"
):
    freq_total = Counter()
    freq_ciencia = Counter()

    for _, row in tqdm(df.iterrows(), total=len(df)):
        dic = asegurar_dict(row[columna_diccionarios])

        if not dic:
            continue

        freq_total.update(dic)

        if row[columna_ciencia] == 1:
            freq_ciencia.update(dic)

    # construir tabla
    data = []
    for palabra, f_total in freq_total.items():
        f_ciencia = freq_ciencia.get(palabra, 0)
        proporcion = f_ciencia / f_total if f_total > 0 else 0

        data.append({
            "palabra": palabra,
            "frecuencia_total": f_total,
            "frecuencia_ciencia": f_ciencia,
            "proporcion_ciencia": proporcion
        })

    df_resultado = pd.DataFrame(data)

    # protección contra DataFrame vacío
    if df_resultado.empty:
        print("⚠️ Advertencia: no se generaron frecuencias.")
        return df_resultado

    return df_resultado.sort_values(
        by="frecuencia_total",
        ascending=False
    ).reset_index(drop=True)



In [23]:
tabla_verbos = construir_tabla_frecuencias(
    df_verbos,
    columna_diccionarios="verbos_lemas_frecuencias"
)

tabla_adjetivos = construir_tabla_frecuencias(
    df_adjetivos,
    columna_diccionarios="adjetivos_lemas_frecuencias"
)

tabla_sustantivos = construir_tabla_frecuencias(
    df_sustantivos,
    columna_diccionarios="sustantivos_lemas_frecuencias"
)

100%|██████████| 62651/62651 [00:11<00:00, 5465.24it/s]


In [36]:
tabla_verbos[
        (tabla_verbos["frecuencia_ciencia"] >= 10) &
        (tabla_verbos["proporcion_ciencia"] >= 0.12)
    ].sort_values("frecuencia_ciencia", ascending=False)

,palabra,frecuencia_total,frecuencia_ciencia,proporcion_ciencia
25,permitir,5243,799,0.152394
20,lograr,5865,738,0.125831
45,generar,3460,735,0.212428
26,buscar,5182,713,0.137592
28,tomar,4935,594,0.120365
...,...,...,...,...
1462,escasear,57,10,0.175439
1795,devenir,39,10,0.256410
1848,comiencer,37,10,0.270270
2190,despliegar,26,10,0.384615


In [38]:
ruta_salida = "/kaggle/working/frecuencias_pos_global_vs_ciencia.xlsx"

with pd.ExcelWriter(ruta_salida) as writer:
    tabla_verbos.to_excel(
        writer,
        sheet_name="verbos",
        index=False
    )
    tabla_adjetivos.to_excel(
        writer,
        sheet_name="adjetivos",
        index=False
    )
    tabla_sustantivos.to_excel(
        writer,
        sheet_name="sustantivos",
        index=False
    )